# Create Tasseled Cap Trends for a small subset in NW Alaska based on Fraser et al. 2014 and Nitze et al 2016

* https://doi.org/10.3390/rs61111533
* https://doi.org/10.1016/j.rse.2016.03.038

## Data and Methods

### Data
* Landsat Surface Reflectance Data
* Try S2 as well?

### Time
* 2005-2024 (Landsat)
* 2015-2024 (S2)
* Only July August

### Location
* 163-162°W; 66.5-67°N

### Filters
< 70% Cloud Cover

### Indices
* Tasseled Cap B-G-W

### Calculate Trend
* create annual mosaics (best method?)
* calculate linear fit for Index values

### Create output map
* 3 bands
  * Band1: slope of TCB
  * Band2: slope of TCG
  * Band3: slope of TCW
  

### Import and EE Intitialization

In [17]:
import ee, eemont, geemap # ee related imports
from tqdm import tqdm # import for progress bar

geemap.ee_initialize(project='water-sinapohlabeln') # this is my ee project, please change to yours

In [18]:
def create_aoi(lon_min:float, lat_min:float, lon_size:float=1, lat_size:float=1) -> ee.Geometry.Polygon:
    """Create EE polygon from coordinates"""
    aoi = ee.Geometry.Polygon(
    [[lon_min, lat_min],
      [lon_min, lat_min+lat_size],
      [lon_min+lon_size, lat_min+lat_size],
      [lon_min+lon_size, lat_min],
      [lon_min, lat_min]])
    return aoi

In [19]:
aoi_test = create_aoi(lon_min=-140, lat_min=65)

In [20]:

# Define the area of interest
aoi = ee.Geometry.Polygon(
    [[[-163, 69],
      [-163, 69.5],
      [-162, 69.5],
      [-162, 69],
      [-163, 69]]])

#aoi = ee.Geometry.Polygon(
#    [[[-163, 66.5],
#      [-163, 67],
#      [-162, 67],
#      [-162, 66.5],
#      [-163, 66.5]]])

# Define the time range
start_date = '2023-07-01'
end_date = '2023-08-31'

#### Example for one year

In [21]:
# Define the maximum cloud cover percentage
max_cloud_cover = 70

# Load Landsat 9 surface reflectance data
l9sr = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2') \
    .filterBounds(aoi) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUD_COVER', max_cloud_cover))
l9_preprocessed = l9sr.preprocess().tasseledCap()
# Load Landsat 9 surface reflectance data
l8sr = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .filterBounds(aoi) \
    .filterDate(start_date, end_date) \
    .filter(ee.Filter.lt('CLOUD_COVER', max_cloud_cover))
    # sacale offset, mask tasseled cap
l8_preprocessed = l8sr.preprocess().tasseledCap()


# Print the image collection
print(l9sr.size().getInfo())

12


Masking


In [ ]:
#def create_snow_mask(image):
#    """
#    Returns a binary snow mask based on NDSI for Landsat 8/9.
#    NDSI = (Green - SWIR1) / (Green + SWIR1)
#    Snow: NDSI > 0.4, NIR > 0.1, Green > 0.11
#    """
#    ndsi = image.normalizedDifference(['SR_B3', 'SR_B6'])  # Green, SWIR1
#    nir = image.select('SR_B5')
#    green = image.select('SR_B3')
#
#    snow = ndsi.gt(0.4).And(nir.gt(0.1)).And(green.gt(0.11))
#    return snow #.Not()




### Create annual Mosaic

In [23]:
def maskLsSr(image):
  cloudShadowBitMask = (1 << 4)
  snowBitMask = (1 << 5)
  cloudsBitMask = (1 << 3)
  # Get the pixel QA band.
  qa = image.select('QA_PIXEL')
  # Both flags should be set to zero, indicating clear conditions.
  mask = qa.bitwiseAnd(cloudShadowBitMask).eq(0) \
                 .And(qa.bitwiseAnd(snowBitMask).eq(0)) \
                 .And(qa.bitwiseAnd(cloudsBitMask).eq(0))
  return image.updateMask(mask)


def create_annual_landsat_mosaic(year, aoi, max_cloud_cover=70):
    """
    Creates an annual mosaic of Landsat 8 and 9 surface reflectance data for a given year and area of interest.

    Args:
        year: The year for which to create the mosaic.
        aoi: The area of interest as an ee.Geometry object.

    Returns:
        An ee.ImageCollection containing the annual mosaics with timestamp information.
    """

    # Define the time range  #S: LANDSAT/LC09/C02/T1_L2
    start_date = f'{year}-07-01'
    end_date = f'{year}-08-31'

    collections =  ['LANDSAT/LC08/C02/T1_L2', 'LANDSAT/LC09/C02/T1_L2'] 

    counter = 0
    collections_list = []
    for collection in collections:
      # Load Landsat
      imCol = ee.ImageCollection(collection) \
          .filterBounds(aoi) \
          .filterDate(start_date, end_date) \
          .filter(ee.Filter.lt('CLOUD_COVER', max_cloud_cover)) \
          .preprocess() \
          .map(maskLsSr) \
          .tasseledCap() \
          .index() \
          
      
      collections_list.append(imCol)

    # Merge all collections
    merged_collection = collections_list[0]
    for col in collections_list[1:]:
      merged_collection = merged_collection.merge(col)


    # Create an annual mosaic
    annual_mosaic = merged_collection.median()
    annual_mosaic = annual_mosaic.addBands(ee.Image.constant(year)\
                                           .rename('year')\
                                           .toDouble()) \
                                           .set({'system:time_start': ee.Date.fromYMD(year, 7, 1).millis()}) \
                                           .clip(aoi)

    return ee.Image(annual_mosaic)



In [ ]:
image_2022 = create_annual_landsat_mosaic(year=2022, aoi=aoi, max_cloud_cover=70)

Create Mosaics over range years and new ImageCollectiom

In [24]:
# create annual mosaic over range of years
mosaic_collection = [create_annual_landsat_mosaic(year, aoi) for year in tqdm(range(2015, 2025))]

# create ee Imagecollection out of it
annual_mosaic = ee.ImageCollection(mosaic_collection)
im = [ee.ImageCollection(annual_mosaic.select(['year', band])).reduce(ee.Reducer.linearFit().unweighted()).select(['scale'], [band]) for band in ['TCB', 'TCG', 'TCW']]
im_full = ee.Image(im).clip(aoi).multiply(10)


100%|██████████| 10/10 [00:19<00:00,  1.91s/it]


Export final Image

In [ ]:

geemap.ee_export_image(im_full, filename='Baldwin_Trend_2015-2024.test1snow.tif', region=aoi, scale=70)


Optional: Überprüfungen

In [ ]:
stats = im_full.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=aoi,
    scale=30,
    maxPixels=1e13
)

print(stats.getInfo())

# Anzahl gültiger Pixel im Trendbild (z.B. auf TCB-Band)
pixel_count = im_full.select('TCB').reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=aoi,
    scale=70,  # Deine Export-Skala
    maxPixels=1e12
)

print("Anzahl gültiger Pixel im Trendbild:", pixel_count.getInfo())

Visualize final Image

In [25]:
m = geemap.Map()
m.add_layer(im_full, dict(min=-0.12, max=0.12))
m.center_object(im_full, 8)
m

Map(center=[69.24976307280684, -162.49999999999991], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
m = geemap.Map()
m.add_layer(image_2022)
m.center_object(image_2022)
m